In [1]:
from pathlib import Path
import re
import shutil
import pandas as pd

# ====== Konfigurasi ======
SPLIT_DIR = Path(r"C:\Users\s2\Documents\FIXRKI\FIXbanget\dataset\split")
CLIENT_DIR = SPLIT_DIR / "clientsMalacious"

NEW_CLIENT_DIR = Path(
    r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V2"
) / "clientsMalacious_Selected"

TOP21_STATES = [5, 23, 41, 46, 13, 54, 17, 40, 16, 20, 51]

SPLITS = ("train", "val")
REQUIRED_FILES = tuple(f"{s}.csv" for s in SPLITS)

SUFFIX = "sinatetic"   # hanya dipakai untuk membersihkan folder lama

COPY_FILES = True
OVERWRITE = False
CLEAN_OLD_SINATETIC = True   # hapus folder state_10xx_sinatetic dari run sebelumnya

if not CLIENT_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {CLIENT_DIR}")

# ====== 1. Deteksi semua state ======
all_states = sorted(
    int(m.group(1))
    for p in CLIENT_DIR.iterdir()
    if p.is_dir() and (m := re.fullmatch(r"state_(\d+)", p.name))
)

NEW_STATES = [s for s in all_states if s not in TOP21_STATES]

print(f"State di disk : {len(all_states)}")
print(f"TOP21         : {len(TOP21_STATES)}")
print(f"State sisa    : {len(NEW_STATES)} -> {NEW_STATES} (tidak disalin)")

# ====== 2. Validasi 21 client terpilih ======
selected_clients = {
    s: CLIENT_DIR / f"state_{s:02d}"
    for s in TOP21_STATES
}

missing = [
    str(path / f)
    for path in selected_clients.values()
    for f in REQUIRED_FILES
    if not (path / f).is_file()
]

if missing:
    raise FileNotFoundError(
        "File berikut tidak ditemukan:\n" + "\n".join(missing)
    )

# ====== 3. Muat data 21 client ======
clients_data = {
    s: {
        split: pd.read_csv(path / f"{split}.csv")
        for split in SPLITS
    }
    for s, path in selected_clients.items()
}

# ====== 4. Ringkasan ======
print(f"\nBerhasil memuat {len(clients_data)} client.\n")

print(f"{'State':<8}{'Train':>10}{'Val':>10}{'Kolom':>8}")

for s, d in clients_data.items():
    print(
        f"{s:02d}".ljust(8)
        + f"{len(d['train']):>10}"
        + f"{len(d['val']):>10}"
        + f"{d['train'].shape[1]:>8}"
    )

# ====== 5. Cek konsistensi kolom ======
ref_cols = list(clients_data[TOP21_STATES[0]]["train"].columns)

for s, d in clients_data.items():
    for split, df in d.items():
        if list(df.columns) != ref_cols:
            print(
                f"⚠️ Kolom state_{s:02d}/{split} "
                "berbeda dari client pertama"
            )

# ====== 6. Salin 21 client TOP21 ke folder Selected ======
NEW_CLIENT_DIR.mkdir(parents=True, exist_ok=True)

# Bersihkan sisa folder _sinatetic dari run sebelumnya
if CLEAN_OLD_SINATETIC:
    old_dirs = list(NEW_CLIENT_DIR.glob(f"state_*_{SUFFIX}"))
    for p in old_dirs:
        shutil.rmtree(p)
    print(f"\nFolder lama dihapus: {len(old_dirs)}")

copied = skipped = absent = 0

for s in TOP21_STATES:
    src_dir = CLIENT_DIR / f"state_{s:02d}"
    dst_dir = NEW_CLIENT_DIR / f"state_{s:02d}"
    dst_dir.mkdir(parents=True, exist_ok=True)

    print(f"state_{s:02d} -> {dst_dir.name}")

    if not COPY_FILES:
        continue

    for f in REQUIRED_FILES:
        src = src_dir / f
        dst = dst_dir / f

        if not src.is_file():
            absent += 1
            print(f"⚠️ Tidak ada di sumber: state_{s:02d}/{f}")
        elif dst.exists() and not OVERWRITE:
            skipped += 1
        else:
            shutil.copy2(src, dst)
            copied += 1

# ====== 7. Verifikasi hasil ======
n_dirs = sum(
    1 for s in TOP21_STATES
    if (NEW_CLIENT_DIR / f"state_{s:02d}").is_dir()
)
n_csv = sum(
    1
    for s in TOP21_STATES
    for f in REQUIRED_FILES
    if (NEW_CLIENT_DIR / f"state_{s:02d}" / f).is_file()
)

# Cek tidak ada folder asing di folder Selected
expected = {f"state_{s:02d}" for s in TOP21_STATES}
extra = sorted(
    p.name for p in NEW_CLIENT_DIR.iterdir()
    if p.is_dir() and p.name not in expected
)

print("\n========== HASIL ==========")
print(f"Folder client baru : {NEW_CLIENT_DIR}")
print(f"Folder state       : {n_dirs} (target {len(TOP21_STATES)})")
print(f"CSV disalin        : {copied}")
print(f"CSV dilewati       : {skipped}")
print(f"CSV tidak tersedia : {absent}")
print(f"Total train/val CSV: {n_csv} (target {len(TOP21_STATES) * len(SPLITS)})")
if extra:
    print(f"⚠️ Folder tak terduga di Selected: {extra}")

State di disk : 44
TOP21         : 11
State sisa    : 33 -> [1, 2, 4, 8, 10, 11, 12, 15, 18, 19, 22, 24, 25, 26, 27, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 44, 45, 48, 49, 50, 53, 55, 56] (tidak disalin)

Berhasil memuat 11 client.

State        Train       Val   Kolom
05            1076       595      72
23            1763      1003      72
41             757       440      72
46            1154       548      72
13            1605       695      72
54            1101       913      72
17             727      1224      72
40            1036       629      72
16             791       283      72
20            1684       914      72
51            1567       673      72

Folder lama dihapus: 0
state_05 -> state_05
state_23 -> state_23
state_41 -> state_41
state_46 -> state_46
state_13 -> state_13
state_54 -> state_54
state_17 -> state_17
state_40 -> state_40
state_16 -> state_16
state_20 -> state_20
state_51 -> state_51

========== HASIL ==========
Folder client baru : C:\Users\s2\Docum

In [2]:
from LabelReverse import reverse_client_labels, compare_outputs

CLIENTS_DIR = r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V2\clientsMalacious_Selected"
NEW_DIR     = r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V2\clientsMalacious_Reverse"

reverse_client_labels(CLIENTS_DIR, label_col="_michd", output_dir=NEW_DIR)
compare_outputs(CLIENTS_DIR, NEW_DIR, label_col="_michd")

state_05
  train.csv: {0: 797, 1: 279} -> {0: 279, 1: 797}
  val.csv: {0: 441, 1: 154} -> {0: 154, 1: 441}
state_13
  train.csv: {0: 1263, 1: 342} -> {0: 342, 1: 1263}
  val.csv: {0: 536, 1: 159} -> {0: 159, 1: 536}
state_16
  train.csv: {0: 635, 1: 156} -> {0: 156, 1: 635}
  val.csv: {0: 220, 1: 63} -> {0: 63, 1: 220}
state_17
  train.csv: {0: 592, 1: 135} -> {0: 135, 1: 592}
  val.csv: {0: 988, 1: 236} -> {0: 236, 1: 988}
state_20
  train.csv: {0: 1321, 1: 363} -> {0: 363, 1: 1321}
  val.csv: {0: 709, 1: 205} -> {0: 205, 1: 709}
state_23
  train.csv: {0: 1322, 1: 441} -> {0: 441, 1: 1322}
  val.csv: {0: 739, 1: 264} -> {0: 264, 1: 739}
state_40
  train.csv: {0: 802, 1: 234} -> {0: 234, 1: 802}
  val.csv: {0: 464, 1: 165} -> {0: 165, 1: 464}
state_41
  train.csv: {0: 615, 1: 142} -> {0: 142, 1: 615}
  val.csv: {0: 336, 1: 104} -> {0: 104, 1: 336}
state_46
  train.csv: {0: 892, 1: 262} -> {0: 262, 1: 892}
  val.csv: {0: 416, 1: 132} -> {0: 132, 1: 416}
state_51
  train.csv: {0: 1235, 1

,state,split,asli_0,asli_1,baru_0,baru_1,semua_baris_terbalik
0,05,train,797,279,279,797,True
1,05,val,441,154,154,441,True
2,13,train,1263,342,342,1263,True
3,13,val,536,159,159,536,True
4,16,train,635,156,156,635,True
5,16,val,220,63,63,220,True
6,17,train,592,135,135,592,True
7,17,val,988,236,236,988,True
8,20,train,1321,363,363,1321,True
9,20,val,709,205,205,709,True


In [4]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V2")
STATE = 5          # harus salah satu dari TOP21_STATES
SUFFIX = ""        # "" -> state_04 | "sinatetic" -> state_04_sinatetic
LABEL = "_michd"

folder = f"state_{STATE:02d}" + (f"_{SUFFIX}" if SUFFIX else "")

path_awal = BASE / "clientsMalacious_Selected" / folder / "train.csv"
path_baru = BASE / "clientsMalacious_Reverse" / folder / "train.csv"

for p in (path_awal, path_baru):
    if not p.is_file():
        raise FileNotFoundError(f"File tidak ditemukan: {p}")

data_awal = pd.read_csv(path_awal)
data_baru = pd.read_csv(path_baru)

print("Data Awal:\n", data_awal[LABEL].head())
print("\nData Baru:\n", data_baru[LABEL].head())

# Verifikasi: semua baris harus terbalik
print("\nJumlah baris sama :", len(data_awal) == len(data_baru))
print("Semua terbalik    :", bool(((data_awal[LABEL] + data_baru[LABEL]) == 1).all()))

Data Awal:
 0    0
1    0
2    0
3    0
4    1
Name: _michd, dtype: int64

Data Baru:
 0    1
1    1
2    1
3    1
4    0
Name: _michd, dtype: int64

Jumlah baris sama : True
Semua terbalik    : True


In [ ]:
print("Data Awal:", data_awal["_michd"].value_counts())
print("Data Baru:", data_baru["_michd"].value_counts())

Data Awal: _michd
0    1390
1     391
Name: count, dtype: int64
Data Baru: _michd
1    1390
0     391
Name: count, dtype: int64
